# SNI-21 Held-out Density Benchmark — B0 sampai B3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ediprin/coffee-bean-detection/blob/agent/add-vadcp-pipeline/notebooks/SNI21_Density_Benchmark_Setup_Colab.ipynb)

Notebook ini membuat benchmark development B0–B3 dari **validation identities saja**.

- B0: 1–5 objek
- B1: 10–25 objek
- B2: 50–100 objek
- B3: 220–300 objek
- Prior utama: empirical source prior
- Visibility utama: mild

**Notebook ini tidak menjalankan training, inference, atau membuka test.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
REPOSITORY = 'https://github.com/ediprin/coffee-bean-detection.git'

if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        REPOSITORY, str(REPO),
    ], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
print('REPO  :', REPO)
print('BRANCH:', BRANCH)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO, check=True)

In [ ]:
CROP_DATASET = Path('/content/drive/MyDrive/coffee-sni-instance-crop-v1')
OUTPUT_ROOT = Path('/content/drive/MyDrive/02_RISET_DAN_PROYEK/Coffee_Bean_Detection/benchmarks/sni21-density-benchmark-v1')
SHARD_CACHE = Path('/content/sni21-shard-cache')

assert CROP_DATASET.is_dir(), f'Dataset crop tidak ditemukan: {CROP_DATASET}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('CROP DATASET:', CROP_DATASET)
print('OUTPUT      :', OUTPUT_ROOT)
print('CACHE       :', SHARD_CACHE)
print('STAGE       : core = B0–B3, empirical prior, mild visibility')

In [ ]:
command = [
    sys.executable, '-u', '-m',
    'coffee_detector.run_sni21_density_benchmark_setup',
    '--crop-dataset-root', str(CROP_DATASET),
    '--output-root', str(OUTPUT_ROOT),
    '--stage', 'core',
    '--scenes-per-condition', '200',
    '--seed', '42',
    '--shard-cache-root', str(SHARD_CACHE),
]

print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
assert return_code == 0, f'Generator gagal dengan return code {return_code}'
print('\nGENERATOR SELESAI')

In [ ]:
from IPython.display import display
import pandas as pd

SUMMARY_PATH = OUTPUT_ROOT / 'setup_core_summary.json'
assert SUMMARY_PATH.is_file(), f'Summary belum ditemukan: {SUMMARY_PATH}'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

rows = []
for arm, info in sorted(summary['arms'].items()):
    audit_path = Path(info['audit'])
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    rows.append({
        'arm': arm,
        'density': str(info['density']),
        'images': audit['synthetic_images'],
        'instances': audit['synthetic_annotations'],
        'geometry_ready': audit['geometry_ready'],
        'warnings': len(audit['warnings']),
        'errors': audit['error_count'],
        'audit_pass': audit['safe_for_training'],
    })

print('=== HELD-OUT DENSITY BENCHMARK ===')
print('Training dijalankan :', summary['training_executed'])
print('Test diakses        :', summary['test_images_accessed'])
print('Development only    :', summary['development_only'])
print('Siap dievaluasi     :', summary['ready_for_evaluation'])
print('Summary             :', SUMMARY_PATH)
display(pd.DataFrame(rows))

assert summary['training_executed'] is False
assert summary['test_images_accessed'] is False
assert summary['ready_for_evaluation'] is True
assert all(row['audit_pass'] for row in rows), 'Ada arm yang gagal audit.'
print('\nPASS: kirim tabel dan setup_core_summary.json sebelum evaluasi YOLO.')